In [ ]:
import sys
sys.path.append("/var/www/python/Prod/nighthawk/")
from nighthawk.data.network.node import Node
node_obj = Node(node_nums=[1213], market='SPP')



node_obj.get_price(
        start_dt=str('2026-02-11'),
        end_dt=str('2026-02-12'),
        component=['LMP'],
        type=['DA', 'RT'],
        granularity='hourly'
    )

,dt,hr,node_num,da_total,rt_total
0,2026-02-11,1,1213,23.798800,19.253000
1,2026-02-11,2,1213,23.498899,16.477301
2,2026-02-11,3,1213,23.520901,18.150000
3,2026-02-11,4,1213,24.929800,19.425600
4,2026-02-11,5,1213,29.884300,27.141899
5,2026-02-11,6,1213,35.217098,31.059299
6,2026-02-11,7,1213,48.698700,37.443001
7,2026-02-11,8,1213,46.068001,29.917999
8,2026-02-11,9,1213,40.639801,24.527901
9,2026-02-11,10,1213,45.872501,30.674000


In [21]:
#!/usr/bin/env python3
"""
SPP Day-Ahead LMP download and processing pipeline (pure Python).

Replaces the PHP flow in SPP_download_DART_LMP.php when mode=da:
  1. Read DA LMP CSVs from SPP FTP in memory (pd.read_csv), process to dataframe.
  2. Merge with node list, upload to spp_lmp.settlement_location_da_hourly via sql_functions.
  3. Update node list (spp_core.spp_settlement_location_node_list) for new settlement locations.
  4. Optionally find (dt, hr) missing in last lookback_days and repopulate by re-downloading those dates.
  5. Update VE clear_price from DA and allocate cleared MW.
  6. Update LMP summary tables (daily, monthly, quarterly, annual for DA and RT last-30-days).

Calling:
  python3 spp_dataDownload_da_lmp.py [start_dt] [end_dt] [lookback_days]
  start_dt, end_dt: optional; date range to download (default: next calendar day).
  lookback_days: optional; days to check for missing (dt, hr) and repopulate (default 7; 0 = skip gap fill).
"""

from __future__ import print_function

import sys
import random
import ftplib
from datetime import datetime, timedelta
from io import BytesIO

import pandas as pd
import pytz

sys.path.append("/var/www/python/Prod/nighthawk/")
from nighthawk.util import sql_functions, dataframe_functions

sys.path.append("/var/www/python/Prod/COMMON/monitoring/notification_system/")
from send_msg_through_slack import send_msg_through_slack

sys.path.append("/var/www/python/Prod/SPP/DataDownload/LMP/")
from spp_update_lmp_summary import run_cron_job_summaries

# Config
FTP_HOST = 'pubftp-mte.itespp.org'
FTP_BASE = '/Markets/DA/LMP_By_SETTLEMENT_LOC'
# Per-day file path under FTP_BASE: YYYY/MM/By_Day/DA-LMP-SL-YYYYMMDD0100.csv
TZ = pytz.timezone('America/Chicago')

LMP_TABLE = 'spp_temp.settlement_location_da_hourly_mte'
NODE_LIST_TABLE = 'spp_core.spp_settlement_location_node_list'
MIN_ROWS_QC = 100  # skip file if too few valid rows
BAA_CSV_COL = 'BAA'  # optional column; present once SPP adds it to the data source
script_name = 'spp_dataDownload_da_lmp.py'
warning_string = ''


def _csv_to_dataframe(io_or_path, source=None):
    """Parse one DA LMP CSV (file-like or path) into a dataframe with dt, hr, settlement_location, da_value, marginalloss_value, congestional_value."""
    df = pd.read_csv(io_or_path)
    df.columns = df.columns.str.strip()
    df = df[df['Settlement Location'].notna() & df['Interval'].notna()].copy()
    df['Interval'] = pd.to_datetime(df['Interval'], errors='coerce')
    df = df[df['Interval'].dt.year > 2000].copy()
    if df.empty:
        return df
    # DA rule: Interval - 1 hour -> dt (date) and hr 1-24 (midnight -> previous day, hr 24)
    shifted = df['Interval'] - pd.Timedelta(hours=1)
    df['dt'] = shifted.dt.strftime('%Y-%m-%d')
    df['hr'] = shifted.dt.hour + 1
    df['settlement_location'] = df['Settlement Location'].astype(str).str.strip()
    df['da_value'] = pd.to_numeric(df['LMP'], errors='coerce').fillna(0).astype(float)
    df['marginalloss_value'] = pd.to_numeric(df['MLC'], errors='coerce').fillna(0).astype(float)
    df['congestional_value'] = pd.to_numeric(df['MCC'], errors='coerce').fillna(0).astype(float)
    cols = ['dt', 'hr', 'settlement_location', 'da_value', 'congestional_value', 'marginalloss_value']
    df['baa_zone'] = df[BAA_CSV_COL].astype(str).str.strip().map({'SPP': 'E', 'SWPW': 'W'}).fillna('SPP')
    cols.append('baa_zone')
    return df[cols]


def fetch_da_lmp_dataframe_from_ftp(start_dt, end_dt):
    """
    Fetch DA LMP CSV for each date in [start_dt, end_dt] from SPP FTP (one file per day:
    .../YYYY/MM/By_Day/DA-LMP-SL-YYYYMMDD0100.csv), read in memory, return one concatenated dataframe.
    """
    start_dt = pd.to_datetime(start_dt).date()
    end_dt = pd.to_datetime(end_dt).date()
    if start_dt > end_dt:
        return pd.DataFrame()

    ftp = ftplib.FTP(FTP_HOST)
    ftp.login('anonymous', 'download')
    frames = []
    for d in pd.date_range(start=start_dt, end=end_dt):
        dt = d.date()
        year, month, day = dt.year, dt.month, dt.day
        file_name = 'DA-LMP-SL-{:04d}{:02d}{:02d}0100.csv'.format(year, month, day)
        remote_dir = '{}/{}/{:02d}/By_Day'.format(FTP_BASE, year, month)
        try:
            ftp.cwd(remote_dir)
            bio = BytesIO()
            ftp.retrbinary('RETR ' + file_name, bio.write)
            bio.seek(0)
            one = _csv_to_dataframe(bio, source=file_name)
            if len(one) >= MIN_ROWS_QC:
                frames.append(one)
                print('Processed {} ({} rows)'.format(file_name, len(one)))
            else:
                print('Skipped {} (too few rows: {})'.format(file_name, len(one)))
        except ftplib.error_perm:
            print('FTP failed {} (By_Day dir)'.format(file_name))
        except Exception as e:
            print('Failed {}: {}'.format(file_name, e))
    ftp.quit()

    if not frames:
        print('No CSV data loaded.')
        return pd.DataFrame()
    out = pd.concat(frames, ignore_index=True)
    out = out.drop_duplicates(subset=['dt', 'hr', 'settlement_location'], keep='last')
    print('Total rows: {}'.format(len(out)))
    return out


def update_da_lmp_from_dataframe(df):
    """
    Merge df with node list (from DB), insert new settlement locations into node list,
    then upload LMP dataframe directly to spp_lmp.settlement_location_da_hourly. No temp table.

    If the source data contains a BAA column (parsed as baa_zone), this function also:
      - Updates baa_zone in NODE_LIST_TABLE for existing nodes where the value has changed.
      - Populates baa_zone when inserting new nodes.
    baa_zone is never written to the LMP tables (sl_rt_5_min / settlement_location_*_hourly).
    """
    if df.empty:
        print('No data to update.')
        return

    has_baa = 'baa_zone' in df.columns

    # Get node list from DB
    node_list = sql_functions.download_df_from_sql_db(
        "SELECT node_num, node_name FROM {}".format(NODE_LIST_TABLE)
    )
    df = df.merge(node_list, left_on='settlement_location', right_on='node_name', how='left')
    df = df.drop(columns=['node_name'], errors='ignore')
    # node_num may be NaN for new locations

    # Rows to upload to LMP table (must have node_num)
    lmp_upload = df.loc[df['node_num'].notna() & (df['node_num'] > 0)].copy()
    if lmp_upload.empty:
        print('No rows with node_num to upload.')
        return
    
    #Align columns to the exact order of LMP_TABLE; fill any unexpected missing columns with NaN.
    lmp_table_cols = sql_functions.download_df_from_sql_db(
        "SELECT * FROM {} LIMIT 1".format(LMP_TABLE)
    ).columns.tolist()
    lmp_upload = lmp_upload[lmp_table_cols]
    sql_functions.replace_into_sql_table(lmp_upload, LMP_TABLE)

    print('CSV to DB update done.')


# def update_bid_prices():
#     """Set clear_price from DA LMP for last 7 days."""
#     sql_functions.run_sql_query("""
#         UPDATE odessa_Bid.SPPFinalBids_multipleStrategies c
#         JOIN spp_lmp.settlement_location_da_hourly da
#           ON c.dt = da.dt AND c.hr = da.hr AND c.node_num = da.node_num
#         SET c.clear_price = da.da_value
#         WHERE da.dt > CURRENT_DATE - INTERVAL 35 DAY
#     """)


# def update_bids_table():
#     """
#     Allocate cleared MW to bids; update odessa_Bid.SPPFinalBids_multipleStrategies.
#     Downloads SPPFinalBids_multipleStrategies (last 30 days) and SPPClearBids JOIN node_list and DA LMP,
#     does allocation in Python, uploads (bid_num, clear_mw) to a temp table, then one UPDATE JOIN to final table.
#     """
#     # 1) Download clear bids with node_name and DA price (one query)
#     clear_df = sql_functions.download_df_from_sql_db("""
#         SELECT c.dt, c.hr, c.nodeNum AS node_num, nl.node_name, c.incdec, c.clearMw AS clear_mw
#         FROM odessa_Bid.SPPClearBids c
#         JOIN spp_lmp.settlement_location_da_hourly da
#           ON c.dt = da.dt AND c.hr = da.hr AND c.nodeNum = da.node_num
#         JOIN spp_core.spp_settlement_location_node_list nl ON c.nodeNum = nl.node_num
#         WHERE c.dt > CURRENT_DATE - INTERVAL 35 DAY AND c.clearMw <> 0
#     """)
#     # 2) Download bids from final table (last 30 days) – columns needed for allocation and for final update
#     bids_df = sql_functions.download_df_from_sql_db("""
#         SELECT bid_num, dt, hr, node_name, incdec, bid_mw, bid_price
#         FROM odessa_Bid.SPPFinalBids_multipleStrategies
#         WHERE dt > CURRENT_DATE - INTERVAL 35 DAY
#     """)
#     if bids_df.empty:
#         print('No bids in last 30 days; skipping bid MW allocation.')
#         return

#     # Normalize dtypes for merging
#     bids_df['dt'] = pd.to_datetime(bids_df['dt']).dt.strftime('%Y-%m-%d')
#     if not clear_df.empty:
#         clear_df['dt'] = pd.to_datetime(clear_df['dt']).dt.strftime('%Y-%m-%d')

#     # 3) Allocation in Python: start with clear_mw = 0 for every bid_num
#     clear_mw_by_bid = dict(zip(bids_df['bid_num'], [0.0] * len(bids_df)))
#     bids_with_mw = bids_df[bids_df['bid_mw'] > 0].copy()

#     for _, r in clear_df.iterrows():
#         dt, hr, node_name, incdec, clear_mw = (
#             r['dt'], int(r['hr']), r['node_name'], r['incdec'], float(r['clear_mw'])
#         )
#         match = (bids_with_mw['dt'] == dt) & (bids_with_mw['hr'] == hr) & \
#                 (bids_with_mw['node_name'].astype(str) == str(node_name)) & (bids_with_mw['incdec'] == incdec)
#         subset = bids_with_mw.loc[match].copy()
#         if subset.empty:
#             continue
#         subset = subset.sort_values('bid_price', ascending=(incdec != 'Decrement'))
#         balance_mw = clear_mw
#         n_bids = len(subset)
#         for idx, b in subset.iterrows():
#             if abs(balance_mw) <= 0.01:
#                 break
#             bid_num, bid_mw = b['bid_num'], float(b['bid_mw'])
#             clear_this = balance_mw if n_bids == 1 else (bid_mw if abs(balance_mw) > bid_mw else balance_mw)
#             clear_mw_by_bid[bid_num] = clear_this
#             balance_mw -= clear_this
#             n_bids -= 1

#     # 4) Build result dataframe: bid_num, clear_mw (all bid_nums in 30-day window)
#     result_df = pd.DataFrame([
#         {'bid_num': bid_num, 'clear_mw': clear_mw_by_bid.get(bid_num, 0.0)}
#         for bid_num in bids_df['bid_num'].unique()
#     ])
#     result_df = result_df.dropna(subset=['bid_num'])

#     # 5) Upload (bid_num, clear_mw) to temp table, then one UPDATE JOIN to final table
#     t_temp_name = 'SPPFinalBids_clear_mw_{}'.format(random.randint(10000, 99999))
#     t_temp = 'temp.{}'.format(t_temp_name)
#     sql_functions.upload_to_sql_from_dataframe(
#         result_df[['bid_num', 'clear_mw']],
#         table_name=t_temp_name,
#         dataset_name='temp',
#     )
#     sql_functions.run_sql_query("""
#         UPDATE odessa_Bid.SPPFinalBids_multipleStrategies b
#         JOIN {} t ON b.bid_num = t.bid_num
#         SET b.clear_mw = t.clear_mw
#     """.format(t_temp))
#     sql_functions.run_sql_query("DROP TABLE IF EXISTS {}".format(t_temp))
#     print('Bid MW allocation done.')


def main():
    timezone = pytz.timezone('America/Chicago')
    now = datetime.now(timezone)
    next_day = (now + timedelta(days=1)).strftime('%Y-%m-%d')

    try:
        start_dt = pd.to_datetime(sys.argv[1]).strftime('%Y-%m-%d')
    except (IndexError, ValueError):
        start_dt = next_day
    try:
        end_dt = pd.to_datetime(sys.argv[2]).strftime('%Y-%m-%d')
    except (IndexError, ValueError):
        end_dt = start_dt

    lookback_days = 7
    if len(sys.argv) > 3:
        try:
            lookback_days = int(sys.argv[3])
        except ValueError:
            pass

    print('start_dt: {}, end_dt: {}, lookback_days (gap fill): {}'.format(start_dt, end_dt, lookback_days))
    if pd.to_datetime(start_dt) > pd.to_datetime(end_dt):
        print('Start date must be <= end date. Exiting.')
        sys.exit(1)

    print('\nGetting LMP data from FTP (in memory)...')
    df = fetch_da_lmp_dataframe_from_ftp('2026-03-15', '2026-03-27')
    print('done')
    print(df.head())

    if not df.empty:
        print('\nUpdating data in tables...')
        update_da_lmp_from_dataframe(df)
        print('done')


    # print('\nSummarizing daily/monthly/etc tables....')
    # run_cron_job_summaries()

if __name__ == '__main__':
    main()


start_dt: 2026-03-28, end_dt: 2026-03-28, lookback_days (gap fill): 7

Getting LMP data from FTP (in memory)...
Processed DA-LMP-SL-202603150100.csv (37872 rows)
Processed DA-LMP-SL-202603160100.csv (37872 rows)
Processed DA-LMP-SL-202603170100.csv (37872 rows)
Processed DA-LMP-SL-202603180100.csv (37872 rows)
Processed DA-LMP-SL-202603190100.csv (37872 rows)
Processed DA-LMP-SL-202603200100.csv (37872 rows)
Processed DA-LMP-SL-202603210100.csv (37872 rows)
Processed DA-LMP-SL-202603220100.csv (37872 rows)
Processed DA-LMP-SL-202603230100.csv (37872 rows)
Processed DA-LMP-SL-202603240100.csv (37872 rows)
Processed DA-LMP-SL-202603250100.csv (37872 rows)
Processed DA-LMP-SL-202603260100.csv (37872 rows)
Processed DA-LMP-SL-202603270100.csv (37872 rows)
Total rows: 492336
done
           dt  hr settlement_location  da_value  congestional_value  \
0  2026-03-15   1                 AEC   20.8181              2.0675   
1  2026-03-15   1           AECC_CSWS   20.8581              2.1771   
2

In [3]:
#!/usr/bin/env python3
"""
SPP Real-Time LMP live-interval download and processing pipeline (pure Python).

Replaces the PHP flow in SPP_download_DART_LMP.php when mode=rt:
  1. Fetch RTBM-LMP-SL-latestInterval.csv from SPP FTP in memory.
  2. Parse CSV: Interval - 5 min gives dt/hr (hr 1-24), forecast_time = raw Interval.
  3. Insert new settlement locations into spp_core.spp_settlement_location_node_list.
  4. REPLACE INTO spp_lmp.sl_rt_5_min (resolved node_nums only).
  5. REPLACE INTO spp_lmp.settlement_location_rt_hourly (avg of 5-min for dt/hr/node).
  6. DELETE from sl_rt_5_min rows older than 14 days.
  7. Find specific forecast_time values missing from spp_lmp.sl_rt_5_min in the last
     lookback_hours and re-download exactly those By_Interval files from the SPP FTP.
     e.g. a missing 2026-03-11 00:45:00 → fetches
     /Markets/RTBM/LMP_By_SETTLEMENT_LOC/2026/03/By_Interval/11/RTBM-LMP-SL-202603110045.csv

For backfilling whole historical dates from the By_Day consolidated files (published ~1 week
later), use spp_dataDownload_daily_rt_lmp.py instead.

Calling:
  python3 spp_dataDownload_rt_lmp.py [lookback_hours]
  lookback_hours : optional; hours to scan for missing intervals (default 4; 0 = skip gap fill).
"""

from __future__ import print_function

import sys
import ftplib
from datetime import datetime, timedelta
from io import BytesIO

import pandas as pd
import pytz

sys.path.append("/var/www/python/Prod/nighthawk/")
from nighthawk.util import sql_functions

sys.path.append("/var/www/python/Prod/COMMON/monitoring/notification_system/")
from send_msg_through_slack import send_msg_through_slack

# Config
FTP_HOST     = 'pubftp-mte.itespp.org'
FTP_RT_BASE  = '/Markets/RTBM/LMP_By_SETTLEMENT_LOC'
FTP_RT_LATEST = FTP_RT_BASE + '/RTBM-LMP-SL-latestInterval.csv'
TZ       = pytz.timezone('America/Chicago')

SL_RT_5MIN_TABLE = 'spp_temp.settlement_location_rt_5_min_mte'
RT_HOURLY_TABLE  = 'spp_temp.settlement_location_rt_hourly_mte'
NODE_LIST_TABLE  = 'spp_core.spp_settlement_location_node_list'

BAA_CSV_COL = 'BAA'  # optional column; present once SPP adds it to the data source
baamap = {'SPP':'E','SWPW':"W"}

script_name    = 'spp_dataDownload_rt_lmp.py'
warning_string = ''


def _csv_to_dataframe(io_or_path):
    """Parse one RT LMP interval CSV into a dataframe."""
    df = pd.read_csv(io_or_path)
    df.columns = df.columns.str.strip()
    df = df[df['Settlement Location'].notna() & df['Interval'].notna()].copy()
    df['Interval'] = pd.to_datetime(df['Interval'], errors='coerce')
    df = df[df['Interval'].dt.year > 2000].copy()
    if df.empty:
        return df

    df['forecast_time']      = df['Interval'].dt.strftime('%Y-%m-%d %H:%M:%S')
    shifted                  = df['Interval'] - pd.Timedelta(minutes=5)
    df['dt']                 = shifted.dt.strftime('%Y-%m-%d')
    df['hr']                 = shifted.dt.hour + 1
    df['settlement_location']= df['Settlement Location'].astype(str).str.strip()
    df['rt_value']           = pd.to_numeric(df['LMP'], errors='coerce').fillna(0).astype(float)
    df['marginalloss_value'] = pd.to_numeric(df['MLC'], errors='coerce').fillna(0).astype(float)
    df['congestional_value'] = pd.to_numeric(df['MCC'], errors='coerce').fillna(0).astype(float)
    if 'BAA' in df.columns:
        df['BAA'] = df['BAA'].map(baamap)
    else:
        df['BAA'] = None

    return df[['forecast_time', 'dt', 'hr', 'settlement_location',
               'rt_value', 'marginalloss_value', 'congestional_value','BAA']]


def fetch_rt_lmp_by_date_range(start_dt, end_dt):
    """
    Fetch all RT LMP 5-min interval CSVs for each day in [start_dt, end_dt].

    For each day, lists and downloads every file under:
      /Markets/RTBM/LMP_By_SETTLEMENT_LOC/YYYY/MM/By_Interval/DD/

    Returns one concatenated, deduplicated dataframe sorted by dt/hr/forecast_time.
    """
    start = pd.to_datetime(start_dt).date()
    end   = pd.to_datetime(end_dt).date()

    ftp = ftplib.FTP(FTP_HOST)
    ftp.login('anonymous', 'download')

    frames = []
    for d in pd.date_range(start=start, end=end):
        dt    = d.date()
        year  = '{:04d}'.format(dt.year)
        month = '{:02d}'.format(dt.month)
        day   = '{:02d}'.format(dt.day)
        remote_dir = '{}/{}/{}/By_Interval/{}'.format(FTP_RT_BASE, year, month, day)

        try:
            ftp.cwd(remote_dir)
            files     = ftp.nlst()
            csv_files = sorted(f for f in files if f.endswith('.csv'))
            print('Day {}: found {} files'.format(dt, len(csv_files)))

            for fname in csv_files:
                try:
                    bio = BytesIO()
                    ftp.retrbinary('RETR ' + fname, bio.write)
                    bio.seek(0)
                    one = _csv_to_dataframe(bio)
                    if not one.empty:
                        frames.append(one)
                except Exception as e:
                    print('  Error reading {}: {}'.format(fname, e))

        except ftplib.error_perm:
            print('Directory not found: {}'.format(remote_dir))
        except Exception as e:
            print('Error listing {}: {}'.format(remote_dir, e))

    ftp.quit()

    if not frames:
        print('No data loaded.')
        return pd.DataFrame()

    out = pd.concat(frames, ignore_index=True)
    # out = out.drop_duplicates(subset=['forecast_time', 'settlement_location'], keep='last')
    out = out.sort_values(['dt', 'hr', 'forecast_time', 'settlement_location']).reset_index(drop=True)
    print('Total: {:,} rows from {} interval files'.format(len(out), len(frames)))
    return out




def update_rt_lmp_from_dataframe(df):
    """
    Merge df with node list, then:
      1. INSERT IGNORE INTO spp_lmp.sl_rt_5_min (resolved node_nums only)
      2. REPLACE INTO settlement_location_rt_hourly (avg by dt/hr/node for last 2 days)
      3. DELETE sl_rt_5_min rows older than 7 days

    Note: settlement_location_rt_5_min (RT_5MIN_TABLE) is intentionally not written.
    The PHP flow confirmed it was dead code — the REPLACE INTO that table was commented
    out in download_LMP_from_CSV_to_DB3.php (2016 revision note: "a new, better indexed
    table sl_rt_5_min replaced it"). Only sl_rt_5_min is the live target.
    """
    if df.empty:
        print('No data to update.')
        return

    node_list = sql_functions.download_df_from_sql_db(
        "SELECT node_num, node_name FROM {}".format(NODE_LIST_TABLE)
    )
    df = df.merge(node_list, left_on='settlement_location', right_on='node_name')
    df = df.drop(columns=['node_name'], errors='ignore')
    print(df.columns)

    # sl_rt_5_min: resolved node_nums only (node_num > 0)
    # sl5_table_cols = sql_functions.download_df_from_sql_db(
    #     "SELECT * FROM {} LIMIT 1".format(SL_RT_5MIN_TABLE)
    # ).columns.tolist()
    if not df.empty:
        sql_functions.replace_into_sql_table(df, SL_RT_5MIN_TABLE)
        print('REPLACE INTO {} done ({} rows).'.format(SL_RT_5MIN_TABLE, len(df)))

    sql_functions.run_sql_query("""
        REPLACE INTO {hourly}
        SELECT rt.dt, rt.hr, rt.node_num, BAA,
               AVG(rt.rt_value)           AS rt_value,
               AVG(rt.congestional_value) AS congestional_value,
               AVG(rt.marginalloss_value) AS marginalloss_value
        FROM {sl5} rt
        GROUP BY rt.dt, rt.hr, rt.node_num
    """.format(hourly=RT_HOURLY_TABLE, sl5=SL_RT_5MIN_TABLE))
    print('REPLACE INTO {} (hourly avg) done.'.format(RT_HOURLY_TABLE))

    # sql_functions.run_sql_query(
    #     "DELETE FROM {} WHERE dt < (CURRENT_DATE - INTERVAL 14 DAY)".format(SL_RT_5MIN_TABLE)
    # )
    print('Cleanup of old 5-min rows done.')

    print('RT CSV to DB update done.')



print('\nGetting RT LMP data from FTP (in memory)...')
df = fetch_rt_lmp_by_date_range('2026-03-15','2026-03-27')
if not df.empty:
    print('\nUpdating data in tables...')
    update_rt_lmp_from_dataframe(df)
    print('done')



Getting RT LMP data from FTP (in memory)...
Day 2026-03-15: found 288 files
Day 2026-03-16: found 288 files
Day 2026-03-17: found 288 files
Day 2026-03-18: found 288 files
Day 2026-03-19: found 288 files
Day 2026-03-20: found 288 files
Day 2026-03-21: found 288 files
Day 2026-03-22: found 288 files
Day 2026-03-23: found 288 files
Day 2026-03-24: found 288 files
Day 2026-03-25: found 288 files
Day 2026-03-26: found 288 files
Day 2026-03-27: found 158 files
Total: 5,702,892 rows from 3614 interval files

Updating data in tables...
Index(['forecast_time', 'dt', 'hr', 'settlement_location', 'rt_value',
       'marginalloss_value', 'congestional_value', 'BAA', 'node_num'],
      dtype='object')
REPLACE INTO spp_temp.settlement_location_rt_5_min_mte done (5681208 rows).
REPLACE INTO spp_temp.settlement_location_rt_hourly_mte (hourly avg) done.
Cleanup of old 5-min rows done.
RT CSV to DB update done.
done


In [1]:
import ftplib
from io import BytesIO
from datetime import date
import pandas as pd

FTP_HOST     = 'pubftp-mte.itespp.org'
FTP_RT_BASE  = '/Markets/RTBM/LMP_By_SETTLEMENT_LOC'


def _csv_to_dataframe(io_or_path):
    """Parse one RT LMP interval CSV into a dataframe."""
    df = pd.read_csv(io_or_path)
    df.columns = df.columns.str.strip()
    df = df[df['Settlement Location'].notna() & df['Interval'].notna()].copy()
    df['Interval'] = pd.to_datetime(df['Interval'], errors='coerce')
    df = df[df['Interval'].dt.year > 2000].copy()
    if df.empty:
        return df

    df['forecast_time']      = df['Interval'].dt.strftime('%Y-%m-%d %H:%M:%S')
    shifted                  = df['Interval'] - pd.Timedelta(minutes=5)
    df['dt']                 = shifted.dt.strftime('%Y-%m-%d')
    df['hr']                 = shifted.dt.hour + 1
    df['settlement_location']= df['Settlement Location'].astype(str).str.strip()
    df['rt_value']           = pd.to_numeric(df['LMP'], errors='coerce').fillna(0).astype(float)
    df['marginalloss_value'] = pd.to_numeric(df['MLC'], errors='coerce').fillna(0).astype(float)
    df['congestional_value'] = pd.to_numeric(df['MCC'], errors='coerce').fillna(0).astype(float)

    return df[['forecast_time', 'dt', 'hr', 'settlement_location',
               'rt_value', 'marginalloss_value', 'congestional_value']]


def fetch_rt_lmp_by_date_range(start_dt, end_dt):
    """
    Fetch all RT LMP 5-min interval CSVs for each day in [start_dt, end_dt].

    For each day, lists and downloads every file under:
      /Markets/RTBM/LMP_By_SETTLEMENT_LOC/YYYY/MM/By_Interval/DD/

    Returns one concatenated, deduplicated dataframe sorted by dt/hr/forecast_time.
    """
    start = pd.to_datetime(start_dt).date()
    end   = pd.to_datetime(end_dt).date()

    ftp = ftplib.FTP(FTP_HOST)
    ftp.login('anonymous', 'download')

    frames = []
    for d in pd.date_range(start=start, end=end):
        dt    = d.date()
        year  = '{:04d}'.format(dt.year)
        month = '{:02d}'.format(dt.month)
        day   = '{:02d}'.format(dt.day)
        remote_dir = '{}/{}/{}/By_Interval/{}'.format(FTP_RT_BASE, year, month, day)

        try:
            ftp.cwd(remote_dir)
            files     = ftp.nlst()
            csv_files = sorted(f for f in files if f.endswith('.csv'))
            print('Day {}: found {} files'.format(dt, len(csv_files)))

            for fname in csv_files:
                try:
                    bio = BytesIO()
                    ftp.retrbinary('RETR ' + fname, bio.write)
                    bio.seek(0)
                    one = _csv_to_dataframe(bio)
                    if not one.empty:
                        frames.append(one)
                except Exception as e:
                    print('  Error reading {}: {}'.format(fname, e))

        except ftplib.error_perm:
            print('Directory not found: {}'.format(remote_dir))
        except Exception as e:
            print('Error listing {}: {}'.format(remote_dir, e))

    ftp.quit()

    if not frames:
        print('No data loaded.')
        return pd.DataFrame()

    out = pd.concat(frames, ignore_index=True)
    out = out.drop_duplicates(subset=['forecast_time', 'settlement_location'], keep='last')
    out = out.sort_values(['dt', 'hr', 'forecast_time', 'settlement_location']).reset_index(drop=True)
    print('Total: {:,} rows from {} interval files'.format(len(out), len(frames)))
    return out


# --- usage ---
start_dt = '2026-03-15'
end_dt   = '2026-03-27'

df = fetch_rt_lmp_by_date_range(start_dt, end_dt)
df.head()


Day 2026-03-15: found 288 files


KeyboardInterrupt: 

In [172]:
# 1. Fetch Source Bids
import os
import sys
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import pytz

import dash
from dash import dcc, html, Input, Output, State, callback, dash_table
import dash_bootstrap_components as dbc
from dash.dash_table.Format import Format, Scheme, Sign, Symbol
from dash.exceptions import PreventUpdate
import plotly.graph_objects as go

sys.path.append("/var/www/python/Prod/nighthawk/")

from nighthawk.util import connections, sql_functions
from nighthawk.data.product.ve import DailyBidsManager
from nighthawk.data.network.node import Node
from nighthawk.viz.giraffe.common_functions import (get_fig, FCC_card, create_title_row, layout_style,
    FCC_datePickerSingle, FCC_input, FCC_dropdown, FCC_button,
    get_callbackFunction_page_usage_counter
)



def simulate_total_ftp(bid_date, sim_end_date,lookback_days, strategy):

    MARKET_CONFIG = {
    'SPP': {
        'load_table': 'spp_physical.spp_latest_load_forecast',
        'wind_table': 'spp_physical.spp_latest_wind_forecast',
        'inc_op_rate': 2.0,
        'dec_op_rate': 0.1
    },
    'PJM': {
        'load_table': '',
        'wind_table': '',
        'inc_op_rate': 0.0,
        'dec_op_rate': 0.0
    },
    'MISO': {'load_table': '', 'wind_table': '', 'inc_op_rate': 0.0, 'dec_op_rate': 0.0},
    'ERCOT': {'load_table': '', 'wind_table': '', 'inc_op_rate': 0.0, 'dec_op_rate': 0.0},
    'NYISO': {'load_table': '', 'wind_table': '', 'inc_op_rate': 0.0, 'dec_op_rate': 0.0},
}
    
    m_cfg = MARKET_CONFIG['SPP']

    strat_list = [strategy] if (strategy and strategy != 'All') else None
    bid_manager = DailyBidsManager(opexchange='SPP', bid_date=str(bid_date), portfolio=pd.DataFrame())
    df_bids = bid_manager.get_bids_from_table(label='current', strategy_ls=strat_list)
    df_bids.rename(columns={'dt': 'source_dt'}, inplace=True)
    expected_cols = ['source_dt', 'hr', 'node_num', 'node_name', 'bid_mw', 'bid_price', 'incdec', 'strategy']
    df_bids = df_bids[[c for c in expected_cols if c in df_bids.columns]]

    if df_bids.empty:
        return [], [], [], [], get_fig(), {"display": "none"}

    df_bids['hr'] = df_bids['hr'].astype(int)
    df_bids['node_num'] = df_bids['node_num'].astype(int)

    # 2. Date Range & Cross Join
    sim_end_obj = pd.to_datetime(sim_end_date).date()
    target_end_date = sim_end_obj 
    target_start_date = sim_end_obj - timedelta(days=int(lookback_days))

    date_range = pd.date_range(start=target_start_date, end=target_end_date, freq='D')
    df_dates = pd.DataFrame({'sim_dt': date_range})
    df_dates['sim_dt_str'] = df_dates['sim_dt'].dt.strftime('%Y-%m-%d')

    df_dates['key'] = 1
    df_bids['key'] = 1
    df_sim = pd.merge(df_dates, df_bids, on='key').drop('key', axis=1)

    # 3. Prices
    node_list = df_bids['node_num'].unique().tolist()

    node_obj = Node(node_nums=node_list, market='SPP')
    df_prices = node_obj.get_price(
        start_dt=str(target_start_date),
        end_dt=str(target_end_date),
        component=['LMP'],
        type=['DA', 'RT'],
        granularity='hourly'
    )

    if not df_prices.empty:
        df_prices.rename(columns={'da_total': 'dalmp', 'rt_total': 'rtlmp'}, inplace=True)
        df_prices['hr'] = df_prices['hr'].astype(int)
        df_prices['node_num'] = df_prices['node_num'].astype(int)
        df_sim = pd.merge(df_sim, df_prices, left_on=['sim_dt_str', 'hr', 'node_num'],
                            right_on=['dt', 'hr', 'node_num'], how='left')
    else:
        df_sim['dalmp'] = np.nan
        df_sim['rtlmp'] = np.nan

    # 4. Calculations (Preserved as per instruction)
    inc_op_rate = m_cfg['inc_op_rate']
    dec_op_rate = m_cfg['dec_op_rate']

    conditions_clear = [
        (df_sim['incdec'] == 'Decrement') & (df_sim['bid_price'] >= df_sim['dalmp']),
        (df_sim['incdec'] == 'Increment') & (df_sim['bid_price'] <= df_sim['dalmp'])
    ]
    df_sim['is_cleared'] = np.select(conditions_clear, [True, True], default=False)
    df_sim.loc[df_sim['dalmp'].isna(), 'is_cleared'] = False
    df_sim['clear_mw'] = np.where(df_sim['is_cleared'], df_sim['bid_mw'], 0.0)

    # DA & RT Cash
    df_sim['total_da_val'] = np.where(df_sim['incdec'] == 'Decrement', -1 * df_sim['clear_mw'] * df_sim['dalmp'], df_sim['clear_mw'] * df_sim['dalmp'])
    df_sim['total_da_elapsed'] = np.where(df_sim['rtlmp'].isna(), 0, df_sim['total_da_val'])

    rt_calc = np.where(df_sim['incdec'] == 'Decrement', df_sim['clear_mw'] * df_sim['rtlmp'], -1 * df_sim['clear_mw'] * df_sim['rtlmp'])
    df_sim['total_rt_elapsed'] = np.where(df_sim['rtlmp'].isna(), 0, rt_calc)

    # RSG
    rsg_cost = np.where(df_sim['incdec'] == 'Increment', df_sim['clear_mw'] * inc_op_rate, df_sim['clear_mw'] * dec_op_rate)
    df_sim['op_rate_val'] = np.where(df_sim['rtlmp'].isna(), 0, -1 * rsg_cost)

    # Totals
    df_sim['gross_pnl'] = df_sim['total_da_elapsed'] + df_sim['total_rt_elapsed']
    df_sim['net_pnl'] = df_sim['gross_pnl'] + df_sim['op_rate_val']
    df_sim.fillna(0, inplace=True)

    # 5. Aggregation & Data Return

    agg_dict = {
        'bid_mw': 'sum', 'clear_mw': 'sum',
        'total_da_val': 'sum', 'total_da_elapsed': 'sum', 'total_rt_elapsed': 'sum', 'op_rate_val': 'sum',
        'gross_pnl': 'sum', 'net_pnl': 'sum'
    }
    renamer = {
        'bid_mw': 'BidMW', 'clear_mw': 'ClearMW',
        'total_da_val': 'TotalDA$', 'total_da_elapsed': 'TotalDAElapsed$',
        'total_rt_elapsed': 'TotalRTElapsed$', 'op_rate_val': 'OpRate$',
        'gross_pnl': 'Gross$', 'net_pnl': 'Net$'
    }

    # A. Overall Summary
    summ_data = df_sim.agg(agg_dict).to_frame().T.rename(columns=renamer).to_dict('records')

    # B. Portfolio Summary
    port_data = df_sim.groupby('strategy').agg(agg_dict).reset_index().rename(columns=renamer).to_dict('records')

    # C. Daily Summary
    daily_df = df_sim.groupby('sim_dt_str').agg(agg_dict).reset_index().rename(columns=renamer)
    daily_df.rename(columns={'sim_dt_str': 'dt'}, inplace=True)
    return df_sim

In [211]:

def simulate_total_mte(bid_date, sim_end_date,lookback_days, strategy):

    MARKET_CONFIG = {
    'SPP': {
        'load_table': 'spp_physical.spp_latest_load_forecast',
        'wind_table': 'spp_physical.spp_latest_wind_forecast',
        'inc_op_rate': 2.0,
        'dec_op_rate': 0.1
    },
    'PJM': {
        'load_table': '',
        'wind_table': '',
        'inc_op_rate': 0.0,
        'dec_op_rate': 0.0
    },
    'MISO': {'load_table': '', 'wind_table': '', 'inc_op_rate': 0.0, 'dec_op_rate': 0.0},
    'ERCOT': {'load_table': '', 'wind_table': '', 'inc_op_rate': 0.0, 'dec_op_rate': 0.0},
    'NYISO': {'load_table': '', 'wind_table': '', 'inc_op_rate': 0.0, 'dec_op_rate': 0.0},
}
    
    m_cfg = MARKET_CONFIG['SPP']

    strat_list = [strategy] if (strategy and strategy != 'All') else None
    bid_manager = DailyBidsManager(opexchange='SPP', bid_date=str(bid_date), portfolio=pd.DataFrame())
    df_bids = bid_manager.get_bids_from_table(label='current', strategy_ls=strat_list)
    df_bids.rename(columns={'dt': 'source_dt'}, inplace=True)
    expected_cols = ['source_dt', 'hr', 'node_num', 'node_name', 'bid_mw', 'bid_price', 'incdec', 'strategy']
    df_bids = df_bids[[c for c in expected_cols if c in df_bids.columns]]

    if df_bids.empty:
        return [], [], [], [], get_fig(), {"display": "none"}

    df_bids['hr'] = df_bids['hr'].astype(int)
    df_bids['node_num'] = df_bids['node_num'].astype(int)


    # 2. Date Range & Cross Join
    sim_end_obj = pd.to_datetime(sim_end_date).date()
    target_end_date = sim_end_obj
    target_start_date = sim_end_obj - timedelta(days=int(lookback_days))
    # print('simulation start date is ', target_start_date)
    # print('simulation end date is ', target_end_date)

    date_range = pd.date_range(start=target_start_date, end=target_end_date, freq='D')
    df_dates = pd.DataFrame({'sim_dt': date_range})
    df_dates['sim_dt_str'] = df_dates['sim_dt'].dt.strftime('%Y-%m-%d')

    df_dates['key'] = 1
    df_bids['key'] = 1
    df_sim = pd.merge(df_dates, df_bids, on='key').drop('key', axis=1)

    # 3. Prices
    
    da_prices = sql_functions.download_df_from_sql_db(f'''select * from spp_temp.settlement_location_da_hourly_mte where dt>='{target_start_date}' and dt <='{target_end_date}' ''')
    rt_prices = sql_functions.download_df_from_sql_db(f'''select * from spp_temp.settlement_location_rt_hourly_mte where dt>='{target_start_date}' and dt <='{target_end_date}' ''')
    da_prices['da_slack']=da_prices['da_value']-da_prices['congestional_value']-da_prices['marginalloss_value']
    rt_prices['rt_slack']=rt_prices['rt_value']-rt_prices['congestional_value']-rt_prices['marginalloss_value']
    df_prices= pd.merge(da_prices, rt_prices, on=['dt', 'hr', 'node_num'], how='left')



    # select_cols= ['dt','hr','settlement_location','da_value','rt_value','congestional_value','marginalloss_value','node_num']
    # df_prices= df_prices[select_cols]
    if not df_prices.empty:
        df_prices.rename(columns={'da_value': 'dalmp', 'rt_value': 'rtlmp'}, inplace=True)
        df_prices['hr'] = df_prices['hr'].astype(int)
        df_prices['node_num'] = df_prices['node_num'].astype(int)
        df_prices['dt'] = df_prices['dt'].astype(str)
        df_sim = pd.merge(df_sim, df_prices, left_on=['sim_dt_str', 'hr', 'node_num'],
                            right_on=['dt', 'hr', 'node_num'], how='left')
    else:
        df_sim['dalmp'] = np.nan
        df_sim['rtlmp'] = np.nan

    # 4. Calculations (Preserved as per instruction)
    inc_op_rate = m_cfg['inc_op_rate']
    dec_op_rate = m_cfg['dec_op_rate']

    conditions_clear = [
        (df_sim['incdec'] == 'Decrement') & (df_sim['bid_price'] >= df_sim['dalmp']),
        (df_sim['incdec'] == 'Increment') & (df_sim['bid_price'] <= df_sim['dalmp'])
    ]
    df_sim['is_cleared'] = np.select(conditions_clear, [True, True], default=False)
    df_sim.loc[df_sim['dalmp'].isna(), 'is_cleared'] = False
    df_sim['clear_mw'] = np.where(df_sim['is_cleared'], df_sim['bid_mw'], 0.0)

    # DA & RT Cash
    df_sim['total_da_val'] = np.where(df_sim['incdec'] == 'Decrement', -1 * df_sim['clear_mw'] * df_sim['dalmp'], df_sim['clear_mw'] * df_sim['dalmp'])
    df_sim['total_da_elapsed'] = np.where(df_sim['rtlmp'].isna(), 0, df_sim['total_da_val'])
    df_sim['total_da_slack'] = np.where(df_sim['incdec'] == 'Decrement', -1 * df_sim['clear_mw'] * df_sim['da_slack'], df_sim['clear_mw'] * df_sim['da_slack'])
    df_sim['total_da_slack_elapsed'] = np.where(df_sim['rtlmp'].isna(), 0, df_sim['total_da_slack'])
    df_sim['total_da_congestional'] = np.where(df_sim['incdec'] == 'Decrement', -1 * df_sim['clear_mw'] * df_sim['congestional_value_x'], df_sim['clear_mw'] * df_sim['congestional_value_x'])
    df_sim['total_da_cong_elapsed'] = np.where(df_sim['rtlmp'].isna(), 0, df_sim['total_da_congestional'])

    rt_calc = np.where(df_sim['incdec'] == 'Decrement', df_sim['clear_mw'] * df_sim['rtlmp'], -1 * df_sim['clear_mw'] * df_sim['rtlmp'])
    df_sim['total_rt_elapsed'] = np.where(df_sim['rtlmp'].isna(), 0, rt_calc)
    rt_calc = np.where(df_sim['incdec'] == 'Decrement', df_sim['clear_mw'] * df_sim['rt_slack'], -1 * df_sim['clear_mw'] * df_sim['rt_slack'])
    df_sim['total_rt_slack_elapsed'] = np.where(df_sim['rtlmp'].isna(), 0, rt_calc)
    rt_calc = np.where(df_sim['incdec'] == 'Decrement', df_sim['clear_mw'] * df_sim['congestional_value_y'], -1 * df_sim['clear_mw'] * df_sim['congestional_value_y'])
    df_sim['total_rt_cong_elapsed'] = np.where(df_sim['rtlmp'].isna(), 0, rt_calc)
    

    # RSG
    rsg_cost = np.where(df_sim['incdec'] == 'Increment', df_sim['clear_mw'] * inc_op_rate, df_sim['clear_mw'] * dec_op_rate)
    df_sim['op_rate_val'] = np.where(df_sim['rtlmp'].isna(), 0, -1 * rsg_cost)

    # Totals
    df_sim['gross_pnl'] = df_sim['total_da_elapsed'] + df_sim['total_rt_elapsed']
    df_sim['net_pnl'] = df_sim['gross_pnl'] + df_sim['op_rate_val']
    df_sim['slack_pnl'] = df_sim['total_da_slack_elapsed']+df_sim['total_rt_slack_elapsed']
    df_sim['cong_pnl'] = df_sim['total_da_cong_elapsed']+df_sim['total_rt_cong_elapsed']
    
    df_sim.fillna(0, inplace=True)
    df_sim.sort_values(by=['gross_pnl'],ascending=True,inplace=True)


    # 5. Aggregation & Data Return

    agg_dict = {
        'bid_mw': 'sum', 'clear_mw': 'sum',
        'total_da_val': 'sum', 'total_da_elapsed': 'sum', 'total_rt_elapsed': 'sum', 'op_rate_val': 'sum',
        'gross_pnl': 'sum', 'net_pnl': 'sum'
    }
    renamer = {
        'bid_mw': 'BidMW', 'clear_mw': 'ClearMW',
        'total_da_val': 'TotalDA$', 'total_da_elapsed': 'TotalDAElapsed$',
        'total_rt_elapsed': 'TotalRTElapsed$', 'op_rate_val': 'OpRate$',
        'gross_pnl': 'Gross$', 'net_pnl': 'Net$'
    }

    # A. Overall Summary
    summ_data = df_sim.agg(agg_dict).to_frame().T.rename(columns=renamer).to_dict('records')

    # B. Portfolio Summary
    port_data = df_sim.groupby('strategy').agg(agg_dict).reset_index().rename(columns=renamer).to_dict('records')

    # C. Daily Summary
    daily_df = df_sim.groupby('sim_dt_str').agg(agg_dict).reset_index().rename(columns=renamer)
    daily_df.rename(columns={'sim_dt_str': 'dt'}, inplace=True)

    return df_sim

In [212]:
simulate_total_mte('2026-03-23','2026-03-23',0,'Darwin')

,sim_dt,sim_dt_str,source_dt,hr,node_num,node_name,bid_mw,bid_price,incdec,strategy,dt,settlement_location,dalmp,congestional_value_x,marginalloss_value_x,baa_zone,da_slack,BAA,rtlmp,congestional_value_y,marginalloss_value_y,rt_slack,is_cleared,clear_mw,total_da_val,total_da_elapsed,total_da_slack,total_da_slack_elapsed,total_da_congestional,total_da_cong_elapsed,total_rt_elapsed,total_rt_slack_elapsed,total_rt_cong_elapsed,op_rate_val,gross_pnl,net_pnl,slack_pnl,cong_pnl
1018,2026-03-23,2026-03-23,2026-03-23,13,1271,SPS.VOLT.0023,4.50,66.50,Increment,Darwin,2026-03-23,SPS.VOLT.0023,77.49,-1.32,4.50,E,74.32,E,776.62,0.13,42.26,734.23,True,4.50,348.70,348.70,334.42,334.42,-5.96,-5.96,"-3,494.79","-3,304.05",-0.59,-9.00,"-3,146.09","-3,155.09","-2,969.63",-6.55
1042,2026-03-23,2026-03-23,2026-03-23,13,1524,SPS.VOLT.0199,4.50,54.10,Increment,Darwin,2026-03-23,SPS.VOLT.0199,81.36,-0.85,7.90,E,74.32,E,779.57,0.13,45.20,734.23,True,4.50,366.13,366.13,334.42,334.42,-3.84,-3.84,"-3,508.07","-3,304.05",-0.59,-9.00,"-3,141.93","-3,150.93","-2,969.63",-4.43
1052,2026-03-23,2026-03-23,2026-03-23,13,1671,KCPL.VOLT.0258,4.50,43.90,Increment,Darwin,2026-03-23,KCPL.VOLT.0258,69.02,-4.13,-1.17,E,74.32,E,753.10,0.13,18.74,734.23,True,4.50,310.58,310.58,334.42,334.42,-18.59,-18.59,"-3,388.95","-3,304.05",-0.59,-9.00,"-3,078.37","-3,087.37","-2,969.63",-19.19
1053,2026-03-23,2026-03-23,2026-03-23,13,1673,KCPL.VOLT.0260,4.50,43.10,Increment,Darwin,2026-03-23,KCPL.VOLT.0260,69.00,-4.12,-1.20,E,74.32,E,751.96,0.13,17.59,734.23,True,4.50,310.50,310.50,334.42,334.42,-18.53,-18.53,"-3,383.80","-3,304.05",-0.59,-9.00,"-3,073.30","-3,082.30","-2,969.63",-19.12
992,2026-03-23,2026-03-23,2026-03-23,13,920,WAUE.MRES.EXR1,4.50,39.50,Increment,Darwin,2026-03-23,WAUE.MRES.EXR1,66.42,-3.69,-4.20,E,74.32,E,748.01,0.13,13.65,734.23,True,4.50,298.90,298.90,334.42,334.42,-16.62,-16.62,"-3,366.06","-3,304.05",-0.60,-9.00,"-3,067.16","-3,076.16","-2,969.62",-17.22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1891,2026-03-23,2026-03-23,2026-03-23,23,1715,CSWS.NUEN.SDM1,4.50,90.90,Increment,Darwin,2026-03-23,CSWS.NUEN.SDM1,128.78,101.59,0.96,E,26.23,E,8.35,-113.61,5.75,116.21,True,4.50,579.49,579.49,118.03,118.03,457.15,457.15,-37.58,-522.92,511.23,-9.00,541.91,532.91,-404.89,968.38
1890,2026-03-23,2026-03-23,2026-03-23,23,1715,CSWS.NUEN.SDM1,4.50,67.60,Increment,Darwin,2026-03-23,CSWS.NUEN.SDM1,128.78,101.59,0.96,E,26.23,E,8.35,-113.61,5.75,116.21,True,4.50,579.49,579.49,118.03,118.03,457.15,457.15,-37.58,-522.92,511.23,-9.00,541.91,532.91,-404.89,968.38
829,2026-03-23,2026-03-23,2026-03-23,10,1715,CSWS.NUEN.SDM1,4.50,46.00,Increment,Darwin,2026-03-23,CSWS.NUEN.SDM1,181.34,101.15,0.02,E,80.17,E,20.83,0.52,-0.77,21.07,True,4.50,816.03,816.03,360.78,360.78,455.18,455.18,-93.72,-94.84,-2.36,-9.00,722.32,713.32,265.95,452.82
1263,2026-03-23,2026-03-23,2026-03-23,16,1745,SPS.CUNHM1.MSR,4.50,42.60,Decrement,Darwin,2026-03-23,SPS.CUNHM1.MSR,42.47,-0.11,4.76,E,37.81,E,557.05,0.00,26.21,530.84,True,4.50,-191.09,-191.09,-170.15,-170.15,0.48,0.48,"2,506.72","2,388.78",0.00,-0.45,"2,315.62","2,315.17","2,218.63",0.48


In [213]:
all_results = []

def simulate_dfs(bid_date):
    for i in bid_date:
        dt_str  = pd.Timestamp(i).strftime('%Y-%m-%d')
        sim_end = (pd.Timestamp(i)).strftime('%Y-%m-%d')

        df1 = simulate_total_ftp(dt_str, sim_end, 0, 'Darwin') \
                .groupby(['sim_dt_str', 'incdec']).sum(numeric_only=True)[['net_pnl']].round(0) \
                .rename(columns={'net_pnl': 'Darwin_net_pnl'}).reset_index()
        
        df2 = simulate_total_ftp(dt_str, sim_end, 0, 'Fourier') \
                .groupby(['sim_dt_str', 'incdec']).sum(numeric_only=True)[['net_pnl']].round(0) \
                .rename(columns={'net_pnl': 'Fourier_net_pnl'}).reset_index()
        
        df3 = simulate_total_mte(dt_str, sim_end, 0, 'Darwin') \
                .groupby(['sim_dt_str', 'incdec']).sum(numeric_only=True)[['net_pnl',"slack_pnl","cong_pnl"]].round(0) \
                .rename(columns={'net_pnl': 'Darwin_net_pnl', "slack_pnl":'Darwin_slack_pnl',"cong_pnl":'Darwin_cong_pnl'}).reset_index()

        df4 = simulate_total_mte(dt_str, sim_end, 0, 'Fourier') \
                .groupby(['sim_dt_str', 'incdec']).sum(numeric_only=True)[['net_pnl',"slack_pnl","cong_pnl"]].round(0) \
                .rename(columns={'net_pnl': 'Fourier_net_pnl', "slack_pnl":'Fourier_slack_pnl',"cong_pnl":'Fourier_cong_pnl'}).reset_index()
        
        
        merged = df1.merge(df2, on=['sim_dt_str', 'incdec'], how='outer') \
                    .merge(df3, on=['sim_dt_str', 'incdec'], how='outer') \
                    .merge(df4, on=['sim_dt_str', 'incdec'], how='outer') \
                    .rename(columns={'sim_dt_str': 'dt'})


        print(f"\n{'─'*50}")
        print(f"Bid date: {dt_str}")
        print(merged.to_string(index=False))
        all_results.append(merged)

    return pd.concat(all_results, ignore_index=True)


# ── Run ───────────────────────────────────────────────────────────────────────
start_dt = '2026-03-15'
end_dt   = '2026-03-27'
all_results = []

for i in pd.date_range(start_dt, end_dt):
    simulate_dfs([i])

finals = pd.concat(all_results, ignore_index=True)

# ── Excel ─────────────────────────────────────────────────────────────────────
output_path = '/var/www/python/Qingcheng/simulation_results.xlsx'
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    finals.to_excel(writer, sheet_name='Darwin_vs_Fourier', index=False)

print(f"\nSaved → {output_path}")
display(finals)



──────────────────────────────────────────────────
Bid date: 2026-03-15
        dt    incdec  Darwin_net_pnl_x  Fourier_net_pnl_x  Darwin_net_pnl_y  Darwin_slack_pnl  Darwin_cong_pnl  Fourier_net_pnl_y  Fourier_slack_pnl  Fourier_cong_pnl
2026-03-15 Decrement         14,380.00          10,744.00          7,018.00         15,080.00        -7,719.00          16,632.00          12,249.00          4,630.00
2026-03-15 Increment         14,657.00            -534.00        -37,576.00        -74,769.00        41,223.00        -160,526.00        -213,794.00         51,742.00

──────────────────────────────────────────────────
Bid date: 2026-03-16
        dt    incdec  Darwin_net_pnl_x  Fourier_net_pnl_x  Darwin_net_pnl_y  Darwin_slack_pnl  Darwin_cong_pnl  Fourier_net_pnl_y  Fourier_slack_pnl  Fourier_cong_pnl
2026-03-16 Decrement          2,663.00          13,376.00        129,997.00         89,154.00        40,227.00          77,313.00          79,722.00         -1,539.00
2026-03-16 Incremen

,dt,incdec,Darwin_net_pnl_x,Fourier_net_pnl_x,Darwin_net_pnl_y,Darwin_slack_pnl,Darwin_cong_pnl,Fourier_net_pnl_y,Fourier_slack_pnl,Fourier_cong_pnl
0,2026-03-15,Decrement,"14,380.00","10,744.00","7,018.00","15,080.00","-7,719.00","16,632.00","12,249.00","4,630.00"
1,2026-03-15,Increment,"14,657.00",-534.00,"-37,576.00","-74,769.00","41,223.00","-160,526.00","-213,794.00","51,742.00"
2,2026-03-16,Decrement,"2,663.00","13,376.00","129,997.00","89,154.00","40,227.00","77,313.00","79,722.00","-1,539.00"
3,2026-03-16,Increment,"-6,149.00","-9,506.00","-45,286.00","-41,993.00","-3,766.00","-220,281.00","-345,709.00","111,391.00"
4,2026-03-17,Decrement,"4,776.00","4,245.00",0.00,0.00,0.00,0.00,0.00,0.00
5,2026-03-17,Increment,"10,103.00","22,193.00","100,385.00","103,922.00",242.00,"265,242.00","247,710.00","30,008.00"
6,2026-03-18,Decrement,"-7,242.00","2,606.00","647,479.00","498,240.00","137,688.00","74,734.00","47,047.00","27,303.00"
7,2026-03-18,Increment,"11,230.00","-28,605.00","25,046.00","-13,187.00","39,952.00","-63,580.00","-126,745.00","61,513.00"
8,2026-03-19,Decrement,"-4,604.00","1,333.00","167,962.00","58,709.00","103,521.00","9,301.00","6,220.00","2,601.00"
9,2026-03-19,Increment,"6,604.00","11,827.00","27,576.00","7,595.00","20,669.00","39,059.00","-11,189.00","50,348.00"


In [ ]:
all_results = []

def simulate_dfs(bid_date):
    global df_node
    for i in bid_date:
        dt_str  = pd.Timestamp(i).strftime('%Y-%m-%d')
        sim_end = (pd.Timestamp(i)).strftime('%Y-%m-%d')

        df1 = simulate_total_ftp(dt_str, sim_end, 0, 'Darwin').merge(df_node, on='node_num') \
                .groupby(['sim_dt_str', 'node_zone']).sum(numeric_only=True)[['net_pnl']].round(0) \
                .rename(columns={'net_pnl': 'Darwin_net_pnl'}).reset_index()
        
        df2 = simulate_total_ftp(dt_str, sim_end, 0, 'Fourier').merge(df_node, on='node_num') \
                .groupby(['sim_dt_str', 'node_zone']).sum(numeric_only=True)[['net_pnl']].round(0) \
                .rename(columns={'net_pnl': 'Fourier_net_pnl'}).reset_index()
        
        df3 = simulate_total_mte(dt_str, sim_end, 0, 'Darwin').merge(df_node, on='node_num')  \
                .groupby(['sim_dt_str', 'node_zone']).sum(numeric_only=True)[['net_pnl',"slack_pnl","cong_pnl"]].round(0) \
                .rename(columns={'net_pnl': 'Darwin_net_pnl', "slack_pnl":'Darwin_slack_pnl',"cong_pnl":'Darwin_cong_pnl'}).reset_index()

        df4 = simulate_total_mte(dt_str, sim_end, 0, 'Fourier').merge(df_node, on='node_num') \
                .groupby(['sim_dt_str', 'node_zone']).sum(numeric_only=True)[['net_pnl',"slack_pnl","cong_pnl"]].round(0) \
                .rename(columns={'net_pnl': 'Fourier_net_pnl', "slack_pnl":'Fourier_slack_pnl',"cong_pnl":'Fourier_cong_pnl'}).reset_index()
        
        
        merged = df1.merge(df2, on=['sim_dt_str', 'node_zone'], how='outer') \
                    .merge(df3, on=['sim_dt_str', 'node_zone'], how='outer') \
                    .merge(df4, on=['sim_dt_str', 'node_zone'], how='outer') \
                    .rename(columns={'sim_dt_str': 'dt'})


        print(f"\n{'─'*50}")
        print(f"Bid date: {dt_str}")
        print(merged.to_string(index=False))
        all_results.append(merged)

    return pd.concat(all_results, ignore_index=True)


# ── Run ───────────────────────────────────────────────────────────────────────
start_dt = '2026-03-15'
end_dt   = '2026-03-27'
all_results = []

df_node = sql_functions.download_df_from_sql_db(
    "select node_num, node_name, node_zone "
    "from spp_core.spp_settlement_location_node_list ")
df_node=df_node[['node_num','node_zone']]

for i in pd.date_range(start_dt, end_dt):
    simulate_dfs([i])

finals = pd.concat(all_results, ignore_index=True)

# ── Excel ─────────────────────────────────────────────────────────────────────
output_path = '/var/www/python/Qingcheng/simulation_results_nodezonal.xlsx'
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    finals.to_excel(writer, sheet_name='Darwin_vs_Fourier', index=False)

print(f"\nSaved → {output_path}")


──────────────────────────────────────────────────
Bid date: 2026-03-15
        dt node_zone  Darwin_net_pnl_x  Fourier_net_pnl_x  Darwin_net_pnl_y  Darwin_slack_pnl  Darwin_cong_pnl  Fourier_net_pnl_y  Fourier_slack_pnl  Fourier_cong_pnl
2026-03-15      AECI           -112.00              11.00          1,356.00         -1,737.00         3,135.00          -3,295.00          -6,249.00          3,124.00
2026-03-15      CSWS          1,852.00           2,562.00         11,331.00         -9,478.00        21,271.00          -7,626.00         -43,482.00         34,308.00
2026-03-15       EDE            588.00                NaN         -1,736.00         -3,231.00         1,851.00                NaN                NaN               NaN
2026-03-15    ERCOTE             48.00             164.00              0.00              0.00             0.00         -65,329.00          -8,062.00        -56,171.00
2026-03-15      KACY          4,008.00               0.00           -804.00         -2,791.0

,dt,node_zone,Darwin_net_pnl_x,Fourier_net_pnl_x,Darwin_net_pnl_y,Darwin_slack_pnl,Darwin_cong_pnl,Fourier_net_pnl_y,Fourier_slack_pnl,Fourier_cong_pnl
0,2026-03-15,AECI,-112.00,11.00,"1,356.00","-1,737.00","3,135.00","-3,295.00","-6,249.00","3,124.00"
1,2026-03-15,CSWS,"1,852.00","2,562.00","11,331.00","-9,478.00","21,271.00","-7,626.00","-43,482.00","34,308.00"
2,2026-03-15,EDE,588.00,NaN,"-1,736.00","-3,231.00","1,851.00",NaN,NaN,NaN
3,2026-03-15,ERCOTE,48.00,164.00,0.00,0.00,0.00,"-65,329.00","-8,062.00","-56,171.00"
4,2026-03-15,KACY,"4,008.00",0.00,-804.00,"-2,791.00","2,092.00",0.00,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...
197,2026-03-27,OPPD,179.00,0.00,"1,064.00","1,331.00",10.00,-18.00,-20.00,4.00
198,2026-03-27,SPS,"1,896.00",0.00,"7,469.00","3,630.00","3,446.00",350.00,451.00,-15.00
199,2026-03-27,WAUE,-952.00,73.00,55.00,72.00,-10.00,"-1,380.00","2,432.00","-3,058.00"
200,2026-03-27,WFEC,0.00,849.00,0.00,0.00,0.00,"4,178.00",777.00,"3,458.00"


In [227]:
def top_bottom_5(df, col):
    top5 = (df.groupby('dt', group_keys=False)
              .apply(lambda x: x.nlargest(5, col))
              .assign(rank_type='top5'))
    bot5 = (df.groupby('dt', group_keys=False)
              .apply(lambda x: x.nsmallest(5, col))
              .assign(rank_type='bottom5'))
    return pd.concat([top5, bot5]).sort_values(['dt', 'rank_type', col], ascending=[True, True, False])[['dt', 'node_zone','rank_type', col]]

darwin_ranks  = top_bottom_5(finals.dropna(), 'Darwin_net_pnl_y')
fourier_ranks = top_bottom_5(finals.dropna(), 'Fourier_net_pnl_y')

In [228]:
darwin_ranks.to_excel("/var/www/python/Qingcheng/simulation_results_zonal_topbottom5_darwin.xlsx")
fourier_ranks.to_excel("/var/www/python/Qingcheng/simulation_results_zonal_topbottom5_four.xlsx")

In [200]:
df=simulate_total_mte("2026-03-17",'2026-03-17',0,'Darwin')
df[(df['incdec']=='Decrement') & (df['bid_price']>df['dalmp'])]

,sim_dt,sim_dt_str,source_dt,hr,node_num,node_name,bid_mw,bid_price,incdec,strategy,dt,settlement_location,dalmp,rtlmp,is_cleared,clear_mw,total_da_val,total_da_elapsed,total_rt_elapsed,op_rate_val,gross_pnl,net_pnl


In [144]:
pd.set_option('display.max_columns', None)
df.groupby(['dt','incdec']).sum(numeric_only=True).round(0)


hr  node_num  bid_mw  bid_price    dalmp     rtlmp  \
dt         incdec                                                             
2026-03-22 Decrement   8083    712324  2048.0     1153.0  22744.0   28340.0   
           Increment  15362   1612185  4133.0    -7762.0  43669.0   16659.0   
2026-03-23 Decrement   8083    712324  2048.0     1153.0  28506.0   84778.0   
           Increment  15362   1612185  4133.0    -7762.0  70593.0  282782.0   
2026-03-24 Decrement   8083    712324  2048.0     1153.0  23515.0   60042.0   
           Increment  15362   1612185  4133.0    -7762.0  57116.0  198475.0   
2026-03-25 Decrement   8083    712324  2048.0     1153.0  22668.0   38386.0   
           Increment  15362   1612185  4133.0    -7762.0  46668.0   54453.0   
2026-03-26 Decrement   8083    712324  2048.0     1153.0  32584.0   42806.0   
           Increment  15362   1612185  4133.0    -7762.0  82885.0   26951.0   

                      is_cleared  clear_mw  total_da_val  total_da_elapsed  \
dt         incdec                                                            
2026-03-22 Decrement          20      71.0       -1368.0           -1368.0   
           Increment        1294    3933.0      135658.0          135658.0   
2026-03-23 Decrement          16      54.0       -1342.0           -1342.0   
           Increment        1330    4046.0      216470.0          216470.0   
2026-03-24 Decrement          32     112.0       -1704.0           -1704.0   
           Increment        1305    3959.0      173733.0          173733.0   
2026-03-25 Decrement           6      21.0        -506.0            -506.0   
           Increment        1317    3999.0      144069.0          144069.0   
2026-03-26 Decrement           1       4.0         -90.0             -90.0   
           Increment        1341    4080.0      249898.0          249898.0   

                      total_rt_elapsed  op_rate_val  gross_pnl   net_pnl  
dt         incdec                                                         
2026-03-22 Decrement            3020.0         -7.0     1653.0    1646.0  
           Increment          -43015.0      -7866.0    92643.0   84777.0  
2026-03-23 Decrement            4625.0         -5.0     3283.0    3277.0  
           Increment         -820206.0      -8091.0  -603736.0 -611828.0  
2026-03-24 Decrement            8892.0        -11.0     7188.0    7177.0  
           Increment         -567352.0      -7918.0  -393619.0 -401537.0  
2026-03-25 Decrement            1465.0         -2.0      958.0     956.0  
           Increment         -157706.0      -7998.0   -13637.0  -21635.0  
2026-03-26 Decrement             163.0         -0.0       73.0      72.0  
           Increment          -83353.0      -8159.0   166545.0  158386.0

In [137]:
# Data Analysis
da_ftp = sql_functions.download_df_from_sql_db(f'''select * from spp_lmp.settlement_location_da_hourly where dt>='2026-03-15'  ''')
rt_ftp = sql_functions.download_df_from_sql_db(f'''select * from spp_lmp.settlement_location_rt_hourly where dt>='2026-03-15' ''')
df_ftp= pd.merge(da_ftp, rt_ftp, on=['dt', 'hr', 'node_num'], how='left')

da_mte = sql_functions.download_df_from_sql_db(f'''select * from spp_temp.settlement_location_da_hourly_mte  ''')
rt_mte = sql_functions.download_df_from_sql_db(f'''select * from spp_temp.settlement_location_rt_hourly_mte  ''')
df_mte= pd.merge(da_mte, rt_mte, on=['dt', 'hr', 'node_num'], how='left')

In [102]:
df_ftp['slack_da']= df_ftp.da_value-df_ftp.congestional_value_x-df_ftp.marginalloss_value_x
df_ftp['slack_rt']= df_ftp.rt_value-df_ftp.congestional_value_y-df_ftp.marginalloss_value_y
df_mte['slack_da']= df_mte.da_value-df_mte.congestional_value_x-df_mte.marginalloss_value_x
df_mte['slack_rt']= df_mte.rt_value-df_mte.congestional_value_y-df_mte.marginalloss_value_y


In [115]:

# ── 1. Null counts per column ──────────────────────────────────────────────
print("=== NULL COUNTS ===")
print("\ndf_ftp nulls:")
print(df_ftp.isnull().sum().to_string())
print("\ndf_mte nulls:")
print(df_mte.isnull().sum())

# ── 2. Unique values per column ────────────────────────────────────────────
print("\n=== UNIQUE VALUES ===")
print("\ndf_ftp unique:")
for col in df_ftp.columns:
    print(f"  {col}: {df_ftp[col].nunique()} unique → {sorted(df_ftp[col].dropna().unique())[:20]}")

print("\ndf_mte unique:")
for col in df_mte.columns:
    print(f"  {col}: {df_mte[col].nunique()} unique → {sorted(df_mte[col].dropna().unique())[:20]}")

# ── 3. Max / Min / Mean ────────────────────────────────────────────────────
print("\n=== MAX / MIN / MEAN ===")
print("\ndf_ftp:")
print(df_ftp.describe().loc[['max', 'min', 'mean']].to_string())
print("\ndf_mte:")
print(df_mte.describe().loc[['max', 'min', 'mean']].to_string())

# ── 4. Row count diff per (dt, hr, nodenum) ────────────────────────────────
print("\n=== ROW COUNT DIFF BY (dt, hr, nodenum) ===")
ftp_counts = df_ftp.groupby(['dt', 'hr', 'nodenum']).size().rename('ftp_count')
mte_counts = df_mte.groupby(['dt', 'hr', 'nodenum']).size().rename('mte_count')

diff = pd.merge(ftp_counts, mte_counts, on=['dt', 'hr', 'nodenum'], how='outer').fillna(0)
diff['ftp_count'] = diff['ftp_count'].astype(int)
diff['mte_count'] = diff['mte_count'].astype(int)
diff['diff'] = diff['ftp_count'] - diff['mte_count']

print(f"\nTotal (dt, hr, nodenum) combos: ftp={len(ftp_counts)}, mte={len(mte_counts)}")
print(f"Combos only in ftp: {(diff['mte_count'] == 0).sum()}")
print(f"Combos only in mte: {(diff['ftp_count'] == 0).sum()}")
print(f"Combos with count mismatch: {(diff['diff'] != 0).sum()}")
print("\nRows with differences:")
print(diff[diff['diff'] != 0].sort_values('diff', ascending=False).to_string())


=== NULL COUNTS ===

df_ftp nulls:
dt                          0
hr                          0
node_num                    0
da_value                    0
congestional_value_x        0
marginalloss_value_x        0
rt_value                41745
congestional_value_y    41745
marginalloss_value_y    41745
slack_da                    0
slack_rt                41745

df_mte nulls:
dt                          0
hr                          0
settlement_location         0
da_value                    0
congestional_value_x        0
marginalloss_value_x        0
baa_zone                    0
node_num                    0
BAA                     15720
rt_value                15720
congestional_value_y    15720
marginalloss_value_y    15720
slack_da                    0
slack_rt                15720
dtype: int64

=== UNIQUE VALUES ===

df_ftp unique:
  dt: 14 unique → [datetime.date(2026, 3, 15), datetime.date(2026, 3, 16), datetime.date(2026, 3, 17), datetime.date(2026, 3, 18), datetime.date(202

KeyError: 'nodenum'